НЕДЕЛЯ 3 : ML практика в PyTorch

ДЕНЬ 1 : Train/Val/Test split

In [ ]:
def create_data(n_samples):
      
    x = torch.randn(n_samples, 1) * 2
    y = torch.sin(x**3) + 0.1*torch.randn(n_samples, 1)
    xy_dataset = TensorDataset(x, y)
    n_train, n_val, n_test = int(0.7*n_samples), int(0.15*n_samples), int(0.15*n_samples)
    xy_train_data, xy_val_data, xy_test_data = random_split(xy_dataset, [n_train, n_val, n_test])
    return xy_train_data, xy_val_data, xy_test_data

def create_model():
    return nn.Sequential(
        nn.Linear(1, 128),  # увеличили с 64 до 128
        nn.Tanh(),
        nn.Linear(128, 64),  # увеличили с 32 до 64
        nn.Tanh(),
        nn.Linear(64, 32),
        nn.Tanh(),
        nn.Linear(32, 1)
    )

def train_model(model, dataset, epochs=300, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    
    data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

    for epoch in range(epochs):
        for batch_idx, (batch_x, batch_y) in enumerate(data_loader):
            # Прямой проход
            y_pred = model(batch_x)
            loss = loss_fn(y_pred, batch_y)

            # Обратный проход    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()   
            
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")    

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import numpy as np
from torch.utils.data import random_split, TensorDataset, DataLoader


n = 1000
train_data, val_data, test_data = create_data(n)
print("Данные инициализированы")
model = create_model()
print("модель инициализирована")
losses = train_model(model, train_data)
print("модель обучена")
loss_fn = nn.MSELoss()
x1, y1 = train_data[:]
y_pred1 = model(x1)
loss1 = loss_fn(y_pred1, y1)
x2, y2 = val_data[:]
y_pred2 = model(x2)
loss2 = loss_fn(y_pred2, y2)
x3, y3 = test_data[:]
y_pred3 = model(x3)
loss3 = loss_fn(y_pred3, y3)
print(f'train_loss = {loss1}')
print(f'val_loss = {loss2}')
print(f'test_loss = {loss3}')


ДЕНЬ 2 : Переобучение (визуализация) График train_loss vs val_loss

In [ ]:
def create_data(n_samples):
      
    x = torch.randn(n_samples, 1) * 2
    y = torch.sin(x*5) + 0.1*torch.randn(n_samples, 1)
    xy_dataset = TensorDataset(x, y)
    n_train, n_val = int(0.8*n_samples), int(0.2*n_samples)
    xy_train_data, xy_val_data = random_split(xy_dataset, [n_train, n_val])
    return xy_train_data, xy_val_data

def model_S():
    return nn.Sequential(
        nn.Linear(1, 16),  
        nn.Tanh(),
        nn.Linear(16, 1),  
    )

def model_L():
    return nn.Sequential(
        nn.Linear(1, 256),  
        nn.Tanh(),
        nn.Linear(256, 1),  
    )


def train_model(model, train_dataset, val_dataset, epochs=500, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    train_losses = []
    val_losses = []
    
    x, y = train_dataset[:]
    x_val, y_val = val_dataset[:]

    for epoch in range(epochs):
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  
        train_losses.append(loss.item())

        y_val_pred = model(x_val)   
        loss_val = loss_fn(y_val_pred, y_val)
        val_losses.append(loss_val.item())

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    return train_losses, val_losses


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import numpy as np
from torch.utils.data import random_split, TensorDataset, DataLoader


n = 100
train_data, val_data= create_data(n)
print("Данные инициализированы")
modelS = model_S()
modelL = model_L()
print("модель инициализирована")
losses_trainS, losses_valS = train_model(modelS, train_data, val_data)
losses_trainL, losses_valL = train_model(modelL, train_data, val_data)
print("модель обучена")

# Визуализация
plt.figure(figsize=(12, 12))

# Loss curves
plt.plot(losses_trainS, 'r-', label='train Small', linewidth=2)
plt.plot(losses_valS, 'g-', label='val Small', linewidth=2)
plt.plot(losses_trainL, 'b-', label='train Large', linewidth=2)
plt.plot(losses_valL, 'y-', label='val Large', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Losses')
plt.legend()
plt.grid(True)


ДЕНЬ 3 : L2 регуляризация. Weight decay в Adam

In [ ]:
def create_data(n_samples):
      
    x = torch.randn(n_samples, 1) * 2
    y = torch.sin(x*5) + 0.1*torch.randn(n_samples, 1)
    xy_dataset = TensorDataset(x, y)
    n_train, n_val = int(0.8*n_samples), int(0.2*n_samples)
    xy_train_data, xy_val_data = random_split(xy_dataset, [n_train, n_val])
    return xy_train_data, xy_val_data

def model_S():
    return nn.Sequential(
        nn.Linear(1, 16),  
        nn.Tanh(),
        nn.Linear(16, 1),  
    )

def model_L():
    return nn.Sequential(
        nn.Linear(1, 256),  
        nn.Tanh(),
        nn.Linear(256, 1),  
    )


def train_model(model, train_dataset, val_dataset, wd, epochs=500, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    train_losses = []
    val_losses = []
    
    x, y = train_dataset[:]
    x_val, y_val = val_dataset[:]

    for epoch in range(epochs):
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  
        train_losses.append(loss.item())

        y_val_pred = model(x_val)   
        loss_val = loss_fn(y_val_pred, y_val)
        val_losses.append(loss_val.item())

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    return train_losses, val_losses


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import random_split, TensorDataset, DataLoader


n = 100
train_data, val_data= create_data(n)
print("Данные инициализированы")
model_no_reg = model_L()
model_L2 = model_L()
print("модели инициализированы")
losses_train, losses_val = train_model(model_no_reg, train_data, val_data, 0.0)
losses_trainL2, losses_valL2 = train_model(model_L2, train_data, val_data, 0.001)
print("модели обучены")

# Визуализация
plt.figure(figsize=(12, 12))

# Loss curves
plt.plot(losses_val, 'g-', label='no reg', linewidth=2)
plt.plot(losses_valL2, 'y-', label='L2 reg', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Losses')
plt.legend()
plt.grid(True)


ДЕНЬ 4: Dropout
Тема дня: Случайное "выключение" нейронов

Создать 2 модели:
    model_no_dropout: Linear(1, 128) → Tanh → Linear(128, 1)
    model_dropout: Linear(1, 128) → Tanh → Dropout(0.3) → Linear(128, 1)


Для каждой модели:
    Optimizer = Adam(model.parameters(), lr=0.01)
    train_losses = []
    val_losses = []
    
    Для epoch в range(500):
        model.train()  # важно для dropout!
        Обучить на train
        
        model.eval()  # важно для dropout!
        Оценить на val
        
        Записать losses
    
    Построить график: epoch vs val_loss (2 линии)

Сравнить: какая модель меньше переобучается?

In [ ]:
def create_data(n_samples):
      
    x = torch.randn(n_samples, 1) * 2
    y = torch.sin(x*5) + 0.1*torch.randn(n_samples, 1)
    xy_dataset = TensorDataset(x, y)
    n_train, n_val = int(0.8*n_samples), int(0.2*n_samples)
    xy_train_data, xy_val_data = random_split(xy_dataset, [n_train, n_val])
    return xy_train_data, xy_val_data

def model_no_drop():
    return nn.Sequential(
        nn.Linear(1, 128),  
        nn.Tanh(),
        nn.Linear(128, 1),  
    )

def model_drop():
    return nn.Sequential(
        nn.Linear(1, 128),  
        nn.Tanh(),
        nn.Dropout(p=0.1),
        nn.Linear(128, 1),  
    )

def train_model(model, train_dataset, val_dataset, epochs=500, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    train_losses = []
    val_losses = []
    
    x, y = train_dataset[:]
    x_val, y_val = val_dataset[:]

    for epoch in range(epochs):
        model.train()  # важно для dropout!
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  
        train_losses.append(loss.item())

        model.eval()  # важно для dropout!
        y_val_pred = model(x_val)   
        loss_val = loss_fn(y_val_pred, y_val)
        val_losses.append(loss_val.item())

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    return train_losses, val_losses

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import random_split, TensorDataset, DataLoader


n = 1000
train_data, val_data= create_data(n)
print("Данные инициализированы")
model_no_drop = model_no_drop()
model_drop = model_drop()
print("модели инициализированы")
losses_train, losses_val = train_model(model_no_drop, train_data, val_data)
losses_trainDrop, losses_valDrop = train_model(model_drop, train_data, val_data)
print("модели обучены")

# Визуализация
plt.figure(figsize=(12, 12))

# Loss curves
plt.plot(losses_val, 'g-', label='no Dropout', linewidth=2)
plt.plot(losses_valDrop, 'y-', label='with Dropout 0.3', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Losses')
plt.legend()
plt.grid(True)


## 📚 ДЕНЬ 5 (12 сентября, 3-4 часа): Кросс-валидация (KFold)

### **Тема дня**: 5-fold cross-validation

**Псевдокод**:
Импортировать KFold из sklearn

Сгенерировать данные: x = randn(500, 1), y = 2*x + 1 + шум

Создать KFold(n_splits=5, shuffle=True)

val_losses = ]

Для train_idx, val_idx в KFold.split(x):
x_train = x[train_idx]
y_train = y[train_idx]

x_val = x[val_idx]
y_val = y[val_idx]

Создать новую модель

Optimizer = Adam(model.parameters(), lr=0.01)

Обучить 200 эпох на train

Оценить на val
Записать val_loss

Вывести: val_losses (5 значений)
Вывести: mean val_loss и std

text


**Необходимые функции**:
- `from sklearn.model_selection import KFold`
- `kf = KFold(n_splits=5, shuffle=True)`
- `for train_idx, val_idx in kf.split(x):`
- Индексация тензора: `x[train_idx]` (работает с numpy array или torch tensor)

**Практика**:
1. Реализовать псевдокод
2. Вывести: `Mean val_loss: X.XX ± X.XX`
3. Полюбопытствовать: зачем 5 folds, а не 2 или 10? (подсказка: баланс между стабильностью и скоростью)

In [ ]:
#import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from numpy.random import randn
from sklearn.model_selection import KFold

def SimpleModel():
    return nn.Sequential(
        nn.Linear(1, 128),  
        nn.Tanh(),
        nn.Linear(128, 1),  
    )
epochs = 200
n_samples = 500
x = np.random.randn(n_samples, 1).astype(np.float32) * 2
y = 2*x +1 + 0.1*np.random.randn(n_samples, 1).astype(np.float32)

kf = KFold(n_splits=5, shuffle=True)
model = SimpleModel()
val_losses = []
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for train_index, val_index in kf.split(x):
    x_train, x_val = x[train_index], x[val_index]
    y_train, y_val = y[train_index], y[val_index]

    for epoch in range(epochs):
        model.train()  # важно для dropout!
        y_pred = model(torch.from_numpy(x))
        loss = loss_fn(y_pred, torch.from_numpy(y))
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  
        
        model.eval()  # важно для dropout!
        y_val_pred = model(torch.from_numpy(x_val))   
        loss_val = loss_fn(y_val_pred, torch.from_numpy(y_val))
        val_losses.append(loss_val.item())

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

print('Средний loss = ', sum(val_losses)/len(val_losses), ' std loss = ', np.std(val_losses))


---

## 📚 ДЕНЬ 6 (13 сентября, 3-4 часа): Метрики (R², RMSE, MAE, MSE)

### **Тема дня**: 4 метрики для оценки качества

**ПсеФункция compute_metrics(y_true, y_pred):
MSE = mean((y_true - y_pred)^2)
RMSE = sqrt(MSE)
MAE = mean(|y_true - y_pred|)

SS_res = sum((y_true - y_pred)^2)
SS_tot = sum((y_true - mean(y_true))^2)
R2 = 1 - SS_res / SS_tot

вернуть MSE, RMSE, MAE, R2вдокод**:


Сгенерировать данные: x = randn(500, 1), y = 2*x + 1 + шум

Разделить на train (400) и test (100)

Создать и обучить модель на train

Оценить на test:
y_pred = model(x_test)
mse, rmse, mae, r2 = compute_metrics(y_test, y_pred)

Вывести все 4 метрики

Сравнить с идеальными значениями:
MSE: 0 (идеал)
RMSE: 0 (идеал)
MAE: 0 (идеал)
R²: 1 (идеал)

text


**Необходимые функции**:
- `torch.mean()` — среднее
- `torch.sqrt()` — квадратный корень
- `torch.abs()` — модуль
- `torch.sum()` — сумма

**Практика**:
1. Реализовать псевдокод
2. Заполнить таблицу:
Метрика | Значение | Идеал | Хорошо ли?
--------|----------|-------|-----------
MSE | 0.XX | 0 | ?
RMSE | 0.XX | 0 | ?
MAE | 0.XX | 0 | ?
R² | 0.XX | 1 | ?

text

3. Вывод: какая метрика лучше всего описывает качество?



In [ ]:
def create_data(n_samples):
      
    x = torch.randn(n_samples, 1) * 2
    y = 2*x + 1+ 0.1*torch.randn(n_samples, 1)
    xy_dataset = TensorDataset(x, y)
    n_train, n_val = int(0.8*n_samples), int(0.2*n_samples)
    xy_train_data, xy_val_data = random_split(xy_dataset, [n_train, n_val])
    return xy_train_data, xy_val_data

def compute_metrics(y_true, y_pred):
    MSE =  torch.mean((y_true - y_pred)**2)
    RMSE = torch.sqrt(MSE)
    MAE = torch.mean(torch.abs(y_true - y_pred))

    SS_res = sum((y_true - y_pred)**2)
    SS_tot = sum((y_true - torch.mean(y_true))**2)
    R2 = 1 - SS_res / SS_tot

    return MSE, RMSE, MAE, R2

def SimpleModel():
    return nn.Sequential(
        nn.Linear(1, 128),  
        nn.Tanh(),
        nn.Linear(128, 1),  
    )

def train_model(model, train_dataset, val_dataset, epochs=500, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
        
    x, y = train_dataset[:]
    x_val, y_val = val_dataset[:]

    for epoch in range(epochs):
        model.train()  # важно для dropout!
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()  
        
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    model.eval()  # важно для dropout!
    y_val_pred = model(x_val)   
    valMSE, valRMSE, valMAE, valR2 = compute_metrics(y_val, y_val_pred)

    return valMSE, valRMSE, valMAE, valR2

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import random_split, TensorDataset, DataLoader


n = 500
train_data, val_data= create_data(n)
print("Данные инициализированы")
modelNN= SimpleModel()
print("модель инициализирована")
MSE1, RMSE1, MAE1, R21 = train_model(modelNN, train_data, val_data)
print("модели обучены")
print('MSE =', MSE1, ' RMSE =', RMSE1, ' MAE =', MAE1, ' R2 =', R21)


---

## 📚 ДЕНЬ 7 (14 сентября, 3-4 часа): Проект недели

### **Тема дня**: Full ML pipeline с кросс-валидацией

**Псевдокод**:
Функция create_data(n_samples):
x = randn(n_samples, 1)
y = 2x2 + sin(x3) + шум
вернуть x, y

Функция create_model():
вернуть Sequential(
Linear(1, 64),
Tanh(),
Linear(64, 32),
Tanh(),
Linear(32, 1)
)

Функция train_and_evaluate(x_train, y_train, x_val, y_val):
model = create_model()
Optimizer = Adam(model.parameters(), lr=0.01, weight_decay=0.01)

train_losses = ]
val_losses = ]

Для epoch в range(300):
model.train()
Обучить на train
Записать train_loss

model.eval()
Оценить на val
Записать val_loss

вернуть model, train_losses, val_losses

Функция compute_metrics(y_true, y_pred):
Вычислить MSE, RMSE, MAE, R² (как в Дне 6)
вернуть все 4

Main
x, y = create_data(500)

kf = KFold(n_splits=5, shuffle=True)

all_val_losses = ]
all_metrics = ]

Для train_idx, val_idx в kf.split(x):
x_train = x[train_idx]
y_train = y[train_idx]
x_val = x[val_idx]
y_val = y[val_idx]

model, train_losses, val_losses = train_and_evaluate(...)

model.eval()
with no_grad():
y_pred = model(x_val)

metrics = compute_metrics(y_val, y_pred)

all_val_losses.append(val_losses[-1])
all_metrics.append(metrics)

Вывести:
Mean val_loss: X.XX ± X.XX
Mean R²: X.XX ± X.XX

Построить график: fold number vs val_loss (5 точек)

text


**Необходимые функции**:
- Всё из предыдущих дней
- `numpy.mean()`, `numpy.std()` для усреднения по folds
- `matplotlib` для графика

**Практика**:
1. Реализовать псевдокод
2. Добиться: Mean R² > 0.95
3. Сохранить график: `plt.savefig('week3_project.png')`
4. **Чекпоинт**: если R² > 0.95 — неделя 3 пройдена успешно! ✅

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt


def create_data(n_samples):
    x = np.random.randn(n_samples, 1).astype(np.float32)
    noise = 0.1 * np.random.randn(n_samples, 1).astype(np.float32)
    y = (2 * x**2 + np.sin(x**3) + noise).astype(np.float32)
    return x, y


def create_model():
    return nn.Sequential(
        nn.Linear(1, 64),
        nn.Tanh(),
        nn.Linear(64, 32),
        nn.Tanh(),
        nn.Linear(32, 1),
    )


def train_and_evaluate(x_train, y_train, x_val, y_val):
    model = create_model()
    optimizer = Adam(model.parameters(), lr=0.01, weight_decay=0.01)
    criterion = nn.MSELoss()

    x_train_t = torch.from_numpy(x_train)
    y_train_t = torch.from_numpy(y_train)
    x_val_t = torch.from_numpy(x_val)
    y_val_t = torch.from_numpy(y_val)

    train_losses = []
    val_losses = []

    for _ in range(300):
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(x_train_t)
        train_loss = criterion(y_pred_train, y_train_t)
        train_loss.backward()
        optimizer.step()
        train_losses.append(train_loss.item())

        model.eval()
        with torch.no_grad():
            y_pred_val = model(x_val_t)
            val_loss = criterion(y_pred_val, y_val_t)
            val_losses.append(val_loss.item())

    return model, train_losses, val_losses


def compute_metrics(y_true, y_pred):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return mse, rmse, mae, r2


def main():
    np.random.seed(42)
    torch.manual_seed(42)

    x, y = create_data(500)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    all_val_losses = []
    all_metrics = []

    for train_idx, val_idx in kf.split(x):
        x_train = x[train_idx]
        y_train = y[train_idx]
        x_val = x[val_idx]
        y_val = y[val_idx]

        model, train_losses, val_losses = train_and_evaluate(
            x_train, y_train, x_val, y_val
        )

        model.eval()
        with torch.no_grad():
            y_pred = model(torch.from_numpy(x_val)).numpy()

        metrics = compute_metrics(y_val, y_pred)

        all_val_losses.append(val_losses[-1])
        all_metrics.append(metrics)

    mean_val_loss = np.mean(all_val_losses)
    std_val_loss = np.std(all_val_losses)

    r2_scores = [metrics[3] for metrics in all_metrics]
    mean_r2 = np.mean(r2_scores)
    std_r2 = np.std(r2_scores)

    print(f"Mean val_loss: {mean_val_loss:.2f} ± {std_val_loss:.2f}")
    print(f"Mean R²: {mean_r2:.2f} ± {std_r2:.2f}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, 6), all_val_losses, "o-", linewidth=2, markersize=8)
    plt.xlabel("Fold number")
    plt.ylabel("Validation loss")
    plt.title("Fold number vs val_loss")
    plt.xticks(range(1, 6))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("fold_val_loss.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle

# ---------- 1. Данные ----------
np.random.seed(42)
n_points = 13
x = np.arange(n_points)

# Немного шума + лёгкий тренд, чтобы картинка была живой
y = 10 + 0.6 * x + np.random.normal(0, 1.2, n_points)

window = 3                     # размер окна
ma_x, ma_y = [], []            # сюда будем складывать точки MA

# ---------- 2. Фигура ----------
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.set_xlim(-1, n_points)
ax.set_ylim(y.min() - 3, y.max() + 3)
ax.set_xlabel("Время (t)", fontsize=12)
ax.set_ylabel("Значение", fontsize=12)
ax.set_title("Скользящее среднее, окно = 3", fontsize=14)
ax.grid(alpha=0.3)

# Исходные точки
ax.scatter(x, y, s=80, color="steelblue", zorder=3, label="Исходные данные")
ax.plot(x, y, color="steelblue", alpha=0.4, zorder=2)

# Линия и точки MA (пустые изначально)
ma_line, = ax.plot([], [], color="crimson", lw=2.5, zorder=4,
                   label="Скользящее среднее")
ma_dots, = ax.plot([], [], "o", color="crimson", ms=8, zorder=5)

# Подсветка точек, попавших в окно
highlight = ax.scatter([], [], s=180, facecolor="none",
                       edgecolor="orange", linewidth=3, zorder=6)

# Прямоугольник — само «окно»
rect = Rectangle((0, y.min() - 3), window - 0.5, (y.max() + 3) - (y.min() - 3),
                 linewidth=2, edgecolor="orange",
                 facecolor="orange", alpha=0.15, zorder=1)
ax.add_patch(rect)

# Текст с формулой среднего (обновляется на каждом кадре)
formula_text = ax.text(0.02, 0.97, "", transform=ax.transAxes,
                       fontsize=13, va="top", family="monospace",
                       bbox=dict(boxstyle="round", fc="white", ec="gray"))

ax.legend(loc="lower right", fontsize=11)

# ---------- 3. Анимация ----------
def update(frame):
    i = frame                      # i — индекс левого края окна
    left = i
    right = i + window             # окно [left, right)

    # сдвигаем прямоугольник
    rect.set_x(left - 0.5)

    # подсвечиваем точки внутри окна
    idx = np.arange(left, right)
    highlight.set_offsets(np.c_[x[idx], y[idx]])

    # считаем среднее
    m = y[idx].mean()

    # добавляем точку MA
    ma_x.append(x[idx].mean())     # центр окна по X
    ma_y.append(m)

    ma_line.set_data(ma_x, ma_y)
    ma_dots.set_data(ma_x, ma_y)

    # текст с формулой
    formula_text.set_text(
        f"Окно: t = {left}..{right-1}\n"
        f"MA = ({y[idx[0]]:.1f} + {y[idx[1]]:.1f} + {y[idx[2]]:.1f}) / 3"
        f" = {m:.2f}"
    )

    return rect, highlight, ma_line, ma_dots, formula_text

frames = n_points - window + 1     # сколько позиций проходит окно
anim = FuncAnimation(fig, update, frames=frames, interval=900,
                     blit=False, repeat=True)

# ---------- 4. Сохранение ----------
# GIF (удобно вставлять в PowerPoint)
anim.save("moving_average.gif", writer=PillowWriter(fps=1.2))

# Если нужен MP4 — раскомментируйте (нужен ffmpeg):
# anim.save("moving_average.mp4", writer="ffmpeg", fps=1.5, dpi=150)

plt.close()
print("Готово: moving_average.gif")

Готово: moving_average.gif
